In [1]:
import os
import sys
from pathlib import Path

# Run everything from the project root, so paths like "data/..." and
# run_experiment()'s "ml/models" resolve to the right folders.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parents[1])
print("Working directory:", Path.cwd())

sys.path.append("ml/training")

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from train_model import run_experiment

Working directory: C:\Users\shame\OneDrive\Desktop\EcoPackAI


In [2]:
materials = pd.read_csv("data/processed/materials.csv")

TARGET = "cost_per_unit_usd"
train_df, test_df = train_test_split(materials, test_size=0.2, random_state=42)

print("Train:", train_df.shape, "Test:", test_df.shape)
print(materials[TARGET].describe().round(2))

Train: (322, 23) Test: (81, 23)
count    403.00
mean       8.94
std        7.09
min        0.21
25%        2.81
50%        3.87
75%       14.65
max       25.95
Name: cost_per_unit_usd, dtype: float64


In [3]:
preprocessor = joblib.load("models/preprocessing/preprocessing_pipeline.pkl")
feature_names = preprocessor.get_feature_names_out()

# Leakage check: neither target may appear among the model's input features
leaky = [f for f in feature_names if "cost_per_unit" in f or "co2_emission_per_kg" in f]
print("Number of features:", len(feature_names))
print("Leaky features found:", leaky)
assert not leaky, "Target leakage detected!"

X_train = preprocessor.transform(train_df)
X_test = preprocessor.transform(test_df)
y_train = train_df[TARGET]
y_test = test_df[TARGET]
print("X_train:", X_train.shape, "X_test:", X_test.shape)

Number of features: 31
Leaky features found: []
X_train: (322, 31) X_test: (81, 31)


In [4]:
RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "random_state": 42,
    "n_jobs": -1,
}

# Convergence check: out-of-bag R2 as the forest grows
for n in [25, 50, 100, 200, 300, 500]:
    rf_check = RandomForestRegressor(**{**RF_PARAMS, "n_estimators": n}, oob_score=True)
    rf_check.fit(X_train, y_train)
    print(f"{n:>4} trees -> OOB R2 = {rf_check.oob_score_:.4f}")

  25 trees -> OOB R2 = 0.9958
  50 trees -> OOB R2 = 0.9958
 100 trees -> OOB R2 = 0.9954
 200 trees -> OOB R2 = 0.9950
 300 trees -> OOB R2 = 0.9948
 500 trees -> OOB R2 = 0.9950


In [5]:
rf = RandomForestRegressor(**RF_PARAMS)

version = run_experiment(
    model=rf,
    model_name="RandomForestRegressor",
    target_name=TARGET,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    dataset_version="v1",
    feature_set_version="v1",
    notes="Production cost model, initial hyperparameters (300 trees, no depth limit), random_state=42.",
)

Experiment logged: RandomForestRegressor__cost_per_unit_usd__v1__20261004_171625
  Model saved to: ml\models\RandomForestRegressor_cost_per_unit_usd_v1.pkl
  Metadata saved to: ml\experiments\metadata\RandomForestRegressor__cost_per_unit_usd__v1__20261004_171625.json
  Test R2: 0.9980, Test MAE: 0.1888, Test RMSE: 0.3101


In [6]:
cv = cross_validate(
    RandomForestRegressor(**RF_PARAMS), X_train, y_train, cv=5,
    scoring=["neg_mean_absolute_error", "neg_root_mean_squared_error", "r2"],
)
fold_scores = {
    "MAE": -cv["test_neg_mean_absolute_error"],
    "RMSE": -cv["test_neg_root_mean_squared_error"],
    "R2": cv["test_r2"],
}

y_pred = rf.predict(X_test)
test_scores = {
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred),
}

for metric in ["MAE", "RMSE", "R2"]:
    print(f"{metric:<5} folds: {np.round(fold_scores[metric], 4)}  "
          f"CV mean: {fold_scores[metric].mean():.4f}  Test: {test_scores[metric]:.4f}")

MAE   folds: [0.3068 0.2505 0.2409 0.1608 0.274 ]  CV mean: 0.2466  Test: 0.1888
RMSE  folds: [0.7381 0.3843 0.3661 0.2622 0.6471]  CV mean: 0.4796  Test: 0.3101
R2    folds: [0.9865 0.9972 0.9974 0.9988 0.9907]  CV mean: 0.9941  Test: 0.9980


In [7]:
baseline = pd.read_csv("ml/metrics/baseline_metrics.csv")
lr = baseline[baseline["model"] == "LinearRegression"].set_index("metric")

# Simplest possible reference: predict the average cost of the material type
type_means = train_df.groupby("material_type")[TARGET].mean()
lookup_pred = test_df["material_type"].map(type_means)
lookup_scores = {
    "MAE": mean_absolute_error(y_test, lookup_pred),
    "RMSE": root_mean_squared_error(y_test, lookup_pred),
    "R2": r2_score(y_test, lookup_pred),
}

comparison = pd.DataFrame({
    "Material-type average": lookup_scores,
    "Linear Regression (baseline)": lr["test_value"],
    "Random Forest": test_scores,
}).T.round(4)
comparison

,MAE,RMSE,R2
Material-type average,1.5484,2.8433,0.8306
Linear Regression (baseline),0.3329,0.4044,0.9966
Random Forest,0.1888,0.3101,0.9980


In [8]:
for metric in ["MAE", "RMSE"]:
    before, after = lr.loc[metric, "test_value"], test_scores[metric]
    print(f"{metric}: {before:.3f} -> {after:.3f}  ({(before - after) / before:.0%} lower error)")

errors = test_df.assign(predicted=y_pred, abs_error=(y_test - y_pred).abs())
errors.groupby("material_type")["abs_error"].agg(["mean", "max", "count"]).round(3)

MAE: 0.333 -> 0.189  (43% lower error)
RMSE: 0.404 -> 0.310  (23% lower error)


,mean,max,count
material_type,,,
Cardboard,0.153,1.101,33
Paper/Bio-Based,0.473,0.972,6
Plastic,0.156,0.654,37
Steel,0.329,0.807,5


In [9]:
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
importance.head(10).round(3)

num__reusability_percent                        0.496
num__waste_reduction_impact_percent             0.286
num__total_material_weight_tons                 0.135
num__annual_usage_units                         0.018
num__recycled_content_percent                   0.014
nom__material_type_Plastic                      0.011
ord__recyclability_category                     0.008
num__carbon_footprint_kg_co2_per_unit           0.008
nom__packaging_type_Steel Racks & Containers    0.007
nom__material_type_Steel                        0.005
dtype: float64

In [10]:
rf_cost_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", rf),
])

joblib.dump(rf_cost_pipeline, "ml/models/rf_cost.joblib")

# Reload as the backend would, and predict straight from raw material rows
reloaded = joblib.load("ml/models/rf_cost.joblib")
raw_pred = reloaded.predict(test_df)
print("Reloaded predictions match:", np.allclose(raw_pred, y_pred))
print("Example:", test_df[["material_id", "material_type", TARGET]].head(3).assign(predicted=raw_pred[:3].round(2)))

Reloaded predictions match: True
Example:     material_id material_type  cost_per_unit_usd  predicted
70     MAT_0071       Plastic               1.47       1.23
281    MAT_0282       Plastic              14.68      14.68
283    MAT_0284       Plastic              12.45      12.38


In [11]:
rows = []
for metric in ["MAE", "RMSE", "R2"]:
    row = {"model": "RandomForestRegressor", "target": TARGET, "metric": metric}
    row.update({f"fold_{i + 1}": round(v, 8) for i, v in enumerate(fold_scores[metric])})
    row["cv_mean"] = round(fold_scores[metric].mean(), 8)
    row["test_value"] = round(test_scores[metric], 4)
    rows.append(row)

rf_metrics = pd.DataFrame(rows)
rf_metrics.to_csv("ml/metrics/rf_cost_metrics.csv", index=False)
rf_metrics

,model,target,metric,fold_1,fold_2,fold_3,fold_4,fold_5,cv_mean,test_value
0,RandomForestRegressor,cost_per_unit_usd,MAE,0.306831,0.250524,0.240885,0.160847,0.274042,0.246626,0.1888
1,RandomForestRegressor,cost_per_unit_usd,RMSE,0.738077,0.384252,0.366115,0.262241,0.647079,0.479553,0.3101
2,RandomForestRegressor,cost_per_unit_usd,R2,0.986460,0.997236,0.997440,0.998826,0.990743,0.994141,0.9980
